# Évaluation de KubeLLM par validité des manifestes

**Contribution C1 — AI-Kube**

Ce notebook remplace l'évaluation par score BLEU, dont trois exécutions
successives sur le modèle de base avaient produit 2,95, 1,31 puis 0,51 —
la génération n'étant pas déterministe.

**Ce qui change ici :**

| | Évaluation BLEU initiale | Cette évaluation |
|---|---|---|
| Métrique | similarité textuelle | validité du manifeste |
| Génération | `do_sample=True`, sans graine | `do_sample=False`, déterministe |
| Jeu de test | 5 exemples | 12 exemples (le jeu complet) |
| Découpage | `train_test_split(seed=42)` | **identique**, reproduit ici |

Le notebook **génère** les manifestes sur GPU et les exporte en JSON.
La **validation** s'effectue ensuite localement, l'API Kubernetes n'étant pas
accessible depuis Colab.


In [ ]:
# ── 1. Environnement ──────────────────────────────────────────────────────
import torch
print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} Go")
else:
    print("⚠ Aucun GPU. Menu Exécution > Modifier le type d'exécution > T4 GPU")


In [ ]:
# ── 2. Bibliothèques ──────────────────────────────────────────────────────
# Versions alignées sur celles de l'entraînement, pour que l'adaptateur
# se recharge sans avertissement de compatibilité.
!pip install -q transformers peft accelerate datasets pyyaml
print("installé")


In [ ]:
# ── 3. Fichiers d'entrée ──────────────────────────────────────────────────
# Téléverser :
#   - kubernetes_pairs.jsonl   (le corpus)
#   - kubellm-v1.zip           (l'adaptateur LoRA)
from google.colab import files
up = files.upload()
print(list(up.keys()))


In [ ]:
# ── 4. Décompression de l'adaptateur ──────────────────────────────────────
import zipfile, os
if os.path.exists("kubellm-v1.zip"):
    with zipfile.ZipFile("kubellm-v1.zip") as z:
        z.extractall(".")
ADAPTER = "./kubellm-v1"
print(sorted(os.listdir(ADAPTER)))


In [ ]:
# ── 5. Jeu de test ────────────────────────────────────────────────────────
# Le découpage reproduit EXACTEMENT celui de l'entraînement : même fonction,
# même proportion, même graine. Les 12 exemples obtenus sont donc ceux que le
# modèle n'a jamais vus — condition nécessaire à une évaluation valide.
import json, yaml
from datasets import Dataset

pairs = []
with open("kubernetes_pairs.jsonl", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            try: pairs.append(json.loads(line))
            except: pass

def format_pair(ex):
    out = ex["output"]
    if isinstance(out, dict):
        try: out = yaml.dump(out, default_flow_style=False, allow_unicode=True)
        except: out = str(out)
    elif not isinstance(out, str):
        out = str(out)
    inp = ex.get("input", "Cluster Kubernetes standard")
    if isinstance(inp, dict): inp = str(inp)
    return {"text": f"### Instruction:\n{ex['instruction']}\n\n### Input:\n{inp}\n\n### Response:\n"}

# On conserve l'index d'origine pour retrouver la référence attendue
indexed = [{**p, "_index": i} for i, p in enumerate(pairs)]
split = Dataset.from_list(indexed).train_test_split(test_size=0.1, seed=42)
test_set = list(split["test"])

print(f"Corpus       : {len(pairs)} paires")
print(f"Entraînement : {len(split['train'])}")
print(f"Test         : {len(test_set)}")
print(f"Indices de test : {sorted(e['_index'] for e in test_set)}")


In [ ]:
# ── 6. Génération ─────────────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch, time

BASE = "microsoft/phi-2"
MAX_NEW = 400          # 200 dans l'évaluation initiale : trop court pour
                       # certains manifestes, qui étaient tronqués

def build_prompt(ex):
    inp = ex.get("input") or "Cluster Kubernetes standard"
    if isinstance(inp, dict): inp = str(inp)
    return (f"### Instruction:\n{ex['instruction']}\n\n"
            f"### Input:\n{inp}\n\n### Response:\n")

def generer_tous(with_adapter, label):
    print(f"\n▶ {label}")
    tok = AutoTokenizer.from_pretrained(ADAPTER if with_adapter else BASE,
                                        trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        BASE, dtype=torch.float16, device_map="auto", trust_remote_code=True)
    if with_adapter:
        model = PeftModel.from_pretrained(model, ADAPTER)
    model.eval()

    sorties = []
    for i, ex in enumerate(test_set):
        t = time.time()
        prompt = build_prompt(ex)
        inputs = tok(prompt, return_tensors="pt", truncation=True,
                     max_length=512).to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=MAX_NEW,
                do_sample=False,               # déterministe : la mesure doit
                                               # être reproductible à l'identique
                pad_token_id=tok.pad_token_id,
                eos_token_id=tok.eos_token_id)
        texte = tok.decode(out[0], skip_special_tokens=True)
        if "### Response:" in texte:
            texte = texte.split("### Response:")[-1]
        sorties.append({
            "index": ex["_index"],
            "instruction": ex["instruction"],
            "expected_kind": (ex["output"] or {}).get("kind") if isinstance(ex["output"], dict) else None,
            "generated": texte.strip(),
        })
        print(f"  [{i+1}/{len(test_set)}] {len(texte)} caractères — {time.time()-t:.0f}s")

    del model
    torch.cuda.empty_cache()
    return sorties

resultats = {}
resultats["base"]    = generer_tous(False, "phi-2 (modèle de base)")
resultats["kubellm"] = generer_tous(True,  "phi-2 + LoRA (KubeLLM-v1)")
print("\nGénération terminée")


In [ ]:
# ── 7. Export ─────────────────────────────────────────────────────────────
import json
from google.colab import files

sortie = {
    "base_model": BASE,
    "test_indices": sorted(e["_index"] for e in test_set),
    "max_new_tokens": MAX_NEW,
    "sampling": "deterministe (do_sample=False)",
    "split": "train_test_split(test_size=0.1, seed=42) — identique a l'entrainement",
    "resultats": resultats,
}
with open("kubellm_generations.json", "w", encoding="utf-8") as f:
    json.dump(sortie, f, ensure_ascii=False, indent=2)

print(f"{sum(len(v) for v in resultats.values())} générations exportées")
files.download("kubellm_generations.json")


## Étape suivante, en local

Placer `kubellm_generations.json` dans `data/observations/`, puis :

```bash
python3 scripts/validate_generations.py
```

Ce script extrait le manifeste de chaque sortie et le soumet à
`kubectl apply --dry-run=server`. Il produit trois taux par modèle :
sortie exploitable, structure complète, manifeste accepté par le serveur.
